# Replication Guide (UI-driven): Compare Feature Explanation Types

This is the UI-driven twin of `feature_explanation_user_study_replication_guide.ipynb`.

The original notebook writes the design by hand — `add_iv`, `add_cv`, `add_dv`,
`set_study_protocol`, and literal participant counts. This one reads all of that
from the study-builder export with `ui_to_code_converter`, so the notebook and the
UI can never drift apart.

## What comes from the JSON

| Step | Original notebook | This notebook |
|---|---|---|
| IVs / CVs / DVs | `study.add_iv(...)` x3 | `study.load_ui_design(...)` |
| Protocol | `set_study_protocol(...)` literal | converted from `procedure` |
| Dataset id | `'wine_quality'` literal | `studyDesign.dataset` |
| Participants / trials | literals in `generate_trials` | `study.trial_settings_from_ui_design()` |

## The design in the export

- **Between participants:** XAI method (`lime`, `shap`, `integrated_gradients`, `input_gradients`) x explanation presence (`tested_w_xai`)
- **Within participants, block-counterbalanced:** explanation type (`none`, `attribution`, `importance`)
- **Primary outcome:** forward-simulation accuracy
- **Virtual baseline:** KNN

This is a **wider** design than the original guide (which fixed the method to LIME
and varied only the type). Everything downstream — explanation generation,
simulation, plotting — is adapted for the extra factor.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'src' / 'api.py').exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.api import xaikitTest
from src.xai_adapter.api import (
    generate_ai_prediction_table,
    generate_xai_explanation_tables,
    init_explanation_run,
)

SEED = 42
BASELINE_MODEL = 'knn'  # knn, decision_tree, logistic_regression, or mlp

# The study-builder export that drives this notebook.
UI_DESIGN_JSON = repo_root / 'src' / 'experiment_planner' / 'example_json_from_ui' / 'experiment-design.json'

OUTPUT_DIR = Path(os.environ.get(
    'XAIKIT_TUTORIAL_OUTPUT_DIR',
    repo_root / 'tutorials' / 'feature_explanation_ui_driven_output',
))

# Execution-only knobs. These are NOT part of the study design: the export plans
# 25 participants per between-cell (200 total), which is a full data collection
# rather than a notebook run. Set PILOT_PARTICIPANTS_PER_CELL = None to simulate
# the planned sample.
PILOT_PARTICIPANTS_PER_CELL = 2
NUM_TRAINING = 6  # >= 1 per within-block; the export has no training-trial field

## 1. Load the study design from the UI export

One call replaces the whole hand-written design block. It converts the export to
the canonical XAIKit config, applies the IVs/CVs/DVs, and stores the protocol.

`prepare_dataset=False` because this study needs an explicit five-feature subset,
which the export does not describe — section 3 does that step by hand.

In [2]:
study = xaikitTest('feature_explanation_ui_driven', output_dir=OUTPUT_DIR)

conversion = study.load_ui_design(
    UI_DESIGN_JSON,
    prepare_dataset=False,
    model_type='mlp',
    seed=SEED,
)

UI design -> trial config: wine_quality_coax_study
  dataset     : wine_quality (model_type=mlp)
  explanations: generated_explanation/de_mlp_wine_quality.csv
  IVs:
    xai_method       type=between  randomization=-     levels=['lime', 'shap', 'integrated_gradients', 'input_gradients']
    tested_w_xai     type=between  randomization=-     levels=[True, False]
    xai_type         type=within   randomization=block levels=['none', 'attribution', 'importance']
  CVs: []
  DVs: ['forward_accuracy', 'decision_time']
  sampling    : 25 participants/between-cell, 10 trials/participant, counterbalancing=auto
  apparatus   : 2 configuration(s) retained (not yet consumed)
    - Configuration 1: mode=ours, 10 instance(s)
    - Configuration 2: mode=ours, 5 instance(s)
  Notes:
    - IV `tested_xai` normalized to `tested_w_xai`.
    - DV `forward_sim` is a known alias for `forward_accuracy`.
    - Dataset `Wine Quality` normalized to `wine_quality`.
    - No explanation CSV in the UI export; ass

### Review what converted — and what did not

The converter never guesses. Anything it could not map is listed in
`conversion.unsupported`, and anything it assumed is in `conversion.notes`.

In [3]:
config = conversion.config

print('IVs')
for name, iv in config['ivs'].items():
    print(f"  {name:<14} {iv['type']:<8} randomization={iv.get('randomization', '-'):<6} {iv['levels']}")
print(f"\nDVs      : {list(config['dvs'])}")
print(f"CVs      : {list(config['cvs'])}")
print(f"Dataset  : {config['dataset']['dataset_id']}")
print(f"Sampling : {config['sampling']}")

print('\nNot supported by the trial API:')
for item in conversion.unsupported:
    print(f'  - {item}')

IVs
  xai_method     between  randomization=-      ['lime', 'shap', 'integrated_gradients', 'input_gradients']
  tested_w_xai   between  randomization=-      [True, False]
  xai_type       within   randomization=block  ['none', 'attribution', 'importance']

DVs      : ['forward_accuracy', 'decision_time']
CVs      : []
Dataset  : wine_quality
Sampling : {'participants_per_between_condition': 25, 'trials_per_participant': 10, 'counterbalancing_strategy': 'auto', 'trial_randomization_strategy': 'balanced', 'shuffle_instances': True, 'instance_wise_explanation': False}

Not supported by the trial API:
  - DV `decision_time`: not in the XAIKit DV support matrix (supported: counterfactual_accuracy, forward_accuracy). Kept in the config so validation reports it -- replace it with a supported DV, or measure it outside XAIKit.
  - apparatus (2 configuration(s)): interface parameters such as showTutorial/showPrediction/userPrediction/focusOnImportant/widgets have no trial-API equivalent yet. Re

### Apparatus: retained, not applied

The export carries two interface configurations. XAIKit has no trial-API
equivalent for interface parameters (`showTutorial`, `showPrediction`,
`userPrediction`, `focusOnImportant`, `widgets`), so the converter keeps them
verbatim rather than silently dropping or half-applying them.

They are available on `study.apparatus` and `config['apparatus']` for whatever
consumes them later — an interface layer, or a manual Qualtrics/web build.

In [4]:
apparatus = pd.DataFrame(study.apparatus)
display(apparatus[['label', 'group', 'mode', 'app_id', 'xai_type', 'xai_method', 'instance_ids']])

print('Raw params retained per configuration:')
for entry in study.apparatus:
    print(f"  {entry['label']}: {entry['params']}")

,label,group,mode,app_id,xai_type,xai_method,instance_ids
0,Configuration 1,All participants,ours,wine_quality,importance,None,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
1,Configuration 2,All participants,ours,wine_quality,attribution,lime,"[1, 2, 3, 4, 5]"


Raw params retained per configuration:
  Configuration 1: {'widgets': '', 'instanceIds': '1-10', 'showTutorial': '0', 'showPrediction': '0', 'xaiType': 'importance', 'appId': 'wine_quality'}
  Configuration 2: {'showTutorial': '0', 'focusOnImportant': '0', 'xaiType': 'attribution', 'userSimulation': '0', 'expMethod': 'lime', 'appId': 'wine_quality', 'instanceIds': '1-5', 'showPrediction': '0', 'userPrediction': '1'}


### Resolve the findings before going further

Validation reported two things about the export. Both are real design issues, not
conversion errors, so they are fixed here in the open:

1. **`decision_time` is not a supported DV.** XAIKit supports `forward_accuracy`
   and `counterfactual_accuracy`. Drop it from the simulated DVs — response time
   is still recorded per trial as `pred_time`, so you can analyse it separately.
2. **`tested_w_xai` is declared between-subjects.** The support matrix expects it
   as a trial-level within IV, which is also what the original guide does and what
   keeps explanation presence balanced inside each participant. Switch it.

The export also omits two control variables this study fixes, so add them.

In [5]:
study.DVs.pop('decision_time', None)

# Explanation presence varies trial by trial, as in the original guide.
study.add_iv('tested_w_xai', 'within', [False, True], randomization='trial')

study.add_cv('user_task', ['forward_simulation'])
study.add_cv('num_attributes_shown', [5])

study.validate_design()


IV configuration:
  xai_method           type=between  randomization=-     levels=['lime', 'shap', 'integrated_gradients', 'input_gradients']
  tested_w_xai         type=within   randomization=trial levels=[False, True]
  xai_type             type=within   randomization=block levels=['none', 'attribution', 'importance']

CVs: ['user_task', 'num_attributes_shown']
DVs: ['forward_accuracy']


({'xai_method': {'type': 'between',
   'levels': ['lime', 'shap', 'integrated_gradients', 'input_gradients']},
  'tested_w_xai': {'type': 'within',
   'levels': [False, True],
   'randomization': 'trial'},
  'xai_type': {'type': 'within',
   'levels': ['none', 'attribution', 'importance'],
   'randomization': 'block'}},
 {'user_task': ['forward_simulation'], 'num_attributes_shown': [5]},
 {'forward_accuracy': ['continuous']})

## 2. Complete the participant protocol

The converter already built the protocol from the export's `procedure` block —
six steps, with kinds inferred (consent / practice / survey / trials / survey /
debrief). Only the pieces the export cannot carry are filled in here.

In [6]:
converted_steps = conversion.protocol['procedure_steps']
for step in converted_steps:
    print(f"  {step['stage']:<12} {step['title']}")

  consent      Welcome & consent
  practice     Training / practice
  survey       Demographics questionnaire
  trials       Main task (trials)
  survey       Post-task questionnaire
  debrief      Debrief


In [7]:
study.set_study_protocol(
    study_title='Learning from feature explanations',
    research_questions=conversion.protocol['research_questions'],
    study_summary='Participants train with feedback, then predict AI outputs with or without an explanation.',
    consent_text='Replace with the approved consent and data-handling information.',
    start_survey_questions=['How familiar are you with AI systems?'],
    end_survey_questions=['How helpful was the display?'],
    procedure_steps=converted_steps,
)

### Review the protocol

Use the editor to revise the participant-facing text before generating trials.

In [8]:
protocol_editor = study.edit_study_protocol()

## 3. Prepare the dataset

The dataset id comes from the export; the five-feature subset and the split are
notebook-level choices the export does not describe.

In [9]:
data = study.prepare_dataset(
    config['dataset']['dataset_id'],
    feature_cols=['Alcohol', 'Sulphates', 'SO2', 'Vinegar Taint', 'pH'],
    rank_features_by_target=False,
    model_type=config['dataset']['model_type'],
    test_size=0.4,
    random_state=SEED,
)

Available training datasets: ['adult', 'breast_cancer', 'cardiotocography', 'forest_cover', 'heart_disease', 'king_county_housing', 'mushrooms', 'prima_diabetes', 'wine_quality']
Dataset   : wine_quality  (1599 rows, 5 model features)
Features  : ['Alcohol', 'Sulphates', 'SO2', 'Vinegar Taint', 'pH']
Encoding  : one-hot
Train set : 959 samples  (60%)
Test set  : 640 samples  (40%)
Class balance (train) -> class 0: 829
Class balance (train) -> class 1: 130
First test instanceIds: [1188, 902, 818, 38, 1255, 496, 947, 861, 839, 1559]


## 4. Train the AI

Train the model whose predictions participants will learn.

In [10]:
study.train_AI_model(
    model_type='mlp',
    target_metric='accuracy',
    target_score=0.90,
    max_epochs=1000,
    check_every_epochs=10,
    batch_size=100,
    verbose=False,
)
display(study.training_summary_table())

,target_metric,target_score,final_score,epochs,batch_size,reached_target,target_accuracy,final_accuracy,model_type,dataset
0,accuracy,0.9,0.903024,330,100,True,0.9,0.903024,mlp,wine_quality


## 5. Create trials from the export's sampling block

`trial_settings_from_ui_design()` returns the `generate_trials(...)` keywords the
export implies — participants per cell, trials per participant, counterbalancing
strategy, seed, and output directory.

Two of them have to be overridden, and both are printed so the difference is visible:

- **Trials per participant.** Balanced randomization needs the trial count to
  divide evenly across every *block-within x trial-within* cell. This design has
  3 explanation types x 2 presence levels = **6 cells**, and the export asks for
  **10** trials — which is not a multiple of 6, so trial generation would raise.
  (The export's own 10-over-3-blocks does not divide either, so this is a conflict
  in the design, not something the within/trial switch introduced.) The count is
  rounded up to the next multiple rather than hardcoded, so it stays correct if you
  change the design and re-export.
- **Pilot size.** The export plans 25 participants per between-cell, which is a
  full data collection rather than a notebook run. Set
  `PILOT_PARTICIPANTS_PER_CELL = None` to simulate the planned sample.

`within_cell_count` is the same helper the converter uses to warn about this at
conversion time — check `conversion.notes`.


In [11]:
from src.experiment_planner.ui_to_code_converter import within_cell_count

trial_settings = study.trial_settings_from_ui_design()
print('From the UI export:', trial_settings)

# Balanced randomization requires a whole number of trials per within-cell.
within_ivs = {n: c['levels'] for n, c in study.iv_config.items() if c['type'] == 'within'}
cells = within_cell_count(study.iv_config)
planned_trials = trial_settings['num_testing']
trial_settings['num_testing'] = -(-planned_trials // cells) * cells

if trial_settings['num_testing'] != planned_trials:
    breakdown = ' x '.join(f'{len(levels)} {name}' for name, levels in within_ivs.items())
    print(
        f"\nAdjusted trials/participant {planned_trials} -> {trial_settings['num_testing']}"
        f" so they divide evenly across {cells} within-subjects cells ({breakdown})."
    )

trial_settings['num_training'] = NUM_TRAINING
trial_settings['balance_by_ai_prediction'] = True
trial_settings['output_dir'] = 'trials'
trial_settings['preview_rows'] = 4
if PILOT_PARTICIPANTS_PER_CELL is not None:
    trial_settings['participants_per_between_condition'] = PILOT_PARTICIPANTS_PER_CELL

print('\nRunning with      :', trial_settings)

trial_result = study.generate_trials(**trial_settings)
trials = pd.DataFrame(trial_result.trials)
display(trials.head())


From the UI export: {'participants_per_between_condition': 25, 'num_testing': 10, 'counterbalancing_strategy': 'auto', 'trial_randomization_strategy': 'balanced', 'instance_wise_explanation': False, 'shuffle_instances': True, 'seed': 42, 'output_dir': 'experiment_output'}
Running with        : {'participants_per_between_condition': 2, 'num_testing': 10, 'counterbalancing_strategy': 'auto', 'trial_randomization_strategy': 'balanced', 'instance_wise_explanation': False, 'shuffle_instances': True, 'seed': 42, 'output_dir': 'trials', 'num_training': 6, 'balance_by_ai_prediction': True, 'preview_rows': 4}


ValueError: trials_per_participant must divide evenly across block x trial-level condition cells for balanced randomization. Got 10 trials for 6 cells.

### Check trial balance

The minimum and maximum should match within each phase and AI label.

In [ ]:
balance_audit = (
    trials.groupby(['participantId', 'phase', 'sampled_ai_prediction'])
    .size().rename('trials').reset_index()
    .groupby(['phase', 'sampled_ai_prediction'])['trials']
    .agg(['min', 'max']).reset_index()
)
display(balance_audit)

## 6. Generate the explanations

The original guide generated one LIME table. This design varies the **method**
between participants, so every method level in the export needs its own table —
and each is then rendered as both explanation types.

In [ ]:
xai_methods = config['ivs']['xai_method']['levels']
print('Methods to generate:', xai_methods)

explanation_present = (
    ~trials['xai_type'].astype(str).str.lower().isin(['none', 'no_xai', 'control'])
    & (
        trials['phase'].eq('training')
        | trials['tested_w_xai'].fillna(False).astype(bool)
    )
)
explanation_ids = list(dict.fromkeys(
    trials.loc[explanation_present, 'instanceId'].astype(int).tolist()
))
print(f'Explanation instances: {len(explanation_ids)}')

In [ ]:
explanation_config = init_explanation_run(
    data=data,
    iv_config={'xai_method': {'levels': xai_methods}},
    trained_ai_model=study.trained_ai_model,
    model_name=study.model_name,
    output_dir=OUTPUT_DIR / 'explanations',
    target=1,
    method_kwargs={'lime': {'num_samples': 1000}},
    instance_ids=explanation_ids,
    predictions_by_instance=study.ai_predictions_by_instance,
)
_, explanation_tables = generate_xai_explanation_tables(explanation_config)
prediction_path, prediction_df = generate_ai_prediction_table(explanation_config)
print(f'Generated {len(explanation_tables)} explanation tables')

### Build both explanation types from each method

Same trick as the original guide: attribution keeps the signed values, importance
takes their absolute values. Doing it per method keeps the type comparison
independent of the algorithm.

Each table is tagged `<method>_<type>` — see the next cell for why.

In [ ]:
typed_tables = []
for method, table in zip(xai_methods, explanation_tables):
    value_cols = [c for c in table if c.startswith('a') and c.endswith('_i')]

    attribution_df = table.copy()
    attribution_df['expMethod'] = f'{method}_attribution'

    importance_df = table.copy()
    importance_df['expMethod'] = f'{method}_importance'
    importance_df[value_cols] = importance_df[value_cols].abs()

    typed_tables += [attribution_df, importance_df]

print('Explanation variants:', [t['expMethod'].iloc[0] for t in typed_tables])

### Point each trial at the right explanation

The executor resolves a trial's explanation through **one** column — it reads
`xai_method`, falling back to `xai_type`. This design uses both factors at once,
so a trial in the `lime` x `importance` cell has to name a single key.

Compose the two into an effective key, keeping the original method in
`xai_method_iv` for analysis. Trials in the `none` block get `none`, which the
executor already understands as "show no explanation".

In [ ]:
trials['xai_method_iv'] = trials['xai_method']

is_none = trials['xai_type'].astype(str).str.lower().isin(['none', 'no_xai', 'control'])
trials['xai_method'] = trials['xai_method_iv'].astype(str) + '_' + trials['xai_type'].astype(str)
trials.loc[is_none, 'xai_method'] = 'none'

study.trials = trials.to_dict('records')
display(trials[['participantId', 'phase', 'xai_method_iv', 'xai_type', 'tested_w_xai', 'xai_method']].head(8))

### Combine the trial information

Store predictions and every explanation variant in one pool for preview and simulation.

In [ ]:
prediction_pool = pd.concat(
    [prediction_df, *typed_tables],
    ignore_index=True,
    sort=False,
)
study.prediction_table_path = prediction_path
study.prediction_table = prediction_df
study.combined_explanations = prediction_pool
display(prediction_pool['expMethod'].value_counts().rename('rows').to_frame())

## 7. Configure the baseline and preview the study

KNN is a nearest-example baseline that learns from the training trials.

Note the export names `CoAX` as its user model (`conversion.config['planner_meta']['user_model']`).
That is planning metadata; this notebook keeps the original guide's KNN baseline so
the two notebooks stay comparable. Switch `BASELINE_MODEL` to change it.

In [ ]:
study.set_cognitive_model(
    cognitive_model_id=BASELINE_MODEL,
    model_kwargs={'n_neighbors': 6} if BASELINE_MODEL == 'knn' else {},
)
print('Export planned user model:', conversion.config['planner_meta']['user_model'])

### Preview one participant

Use an attribution trial to check signed explanations and explanation presence.

In [ ]:
preview_id = int(trials.loc[trials['xai_type'].eq('attribution'), 'participantId'].iloc[0])
participant_trials = study.preview_participant_trials(
    participant_id=preview_id,
    visualization='influence',
    class_labels=['Type 1', 'Type 2'],
    fallback='html',
)

## 8. Run the virtual study

Run every participant-condition and keep the testing responses for analysis.

In [ ]:
simulated_results = study.run_experiment(
    mode='whole_experiment',
    participant_id=None,
    explanation_pool=prediction_pool,
)
testing_results = simulated_results.query("phase == 'testing'").copy()
display(testing_results.head())

### Save the results

Export the trial-level responses before plotting.

In [ ]:
csv_path, json_path = study.save_results(out_dir='simulated_results')
print(f'Saved {len(simulated_results):,} responses to {csv_path} and {json_path}')

## 9. Plot the DV against both IVs

Explanation type on the x-axis, XAI method as the bar colour — the two factors the
export set out to compare.

In [ ]:
from src.result_visualizer import plot_dv_by_two_ivs

accuracy_plot = plot_dv_by_two_ivs(
    testing_results,
    x_iv='xai_type',
    hue_iv='xai_method_iv',
    dv='forward_accuracy',
    phase='testing',
    x_levels=config['ivs']['xai_type']['levels'],
    hue_levels=config['ivs']['xai_method']['levels'],
    x_labels={'none': 'None', 'attribution': 'Attribution', 'importance': 'Importance'},
    hue_labels={
        'lime': 'LIME',
        'shap': 'SHAP',
        'integrated_gradients': 'Integrated Gradients',
        'input_gradients': 'Input Gradients',
    },
    title='Forward accuracy by explanation type and XAI method',
)
accuracy_plot.figure;

## Keeping the notebook and the UI in sync

Change the design in the study builder, re-export, and re-run from section 1 —
nothing in the design cells is hand-written, so the notebook follows.

The three things this notebook still specifies itself, because the export does not
carry them, are: the five-feature subset and split (section 3), the training-trial
count and pilot size (section 5), and the effective-explanation-key composition
(section 6). Each is flagged where it happens.